# Exp 6 - CAN Bus Arbitration Simulation using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Simulate CAN bus arbitration and show why lower CAN identifiers should be assigned to higher-priority messages.

CAN arbitration is message-priority based. When multiple nodes attempt to transmit, the message with the lower numerical identifier wins arbitration and continues transmission.

## Textbook Notes and Case Studies

### 1. Textbook Background

Controller Area Network is a broadcast bus widely used in vehicles and embedded systems. A key idea in CAN is message-based priority arbitration. Instead of assigning priority to a node, CAN assigns priority to the message identifier. Lower numerical identifier values represent higher priority in standard CAN arbitration.

The arbitration process is non-destructive: if multiple nodes begin transmitting at the same time, the highest-priority frame continues while lower-priority transmitters stop and retry later. This is important for real-time communication because urgent messages can win bus access without corrupting the winning frame.

### 2. Architecture Notes

```
Node A Message ID ----Node B Message ID -----+--> Shared CAN Bus Arbitration -> Winning Frame -> All Nodes Receive
Node C Message ID ----/
                                |
                                v
                         Lower ID Wins Priority
```

CAN is a multi-master bus. Any node may attempt transmission when the bus is idle. Arbitration compares identifier bits. Dominant bits override recessive bits, so the node transmitting the lower identifier remains in control.

### 3. Important Formulas

Priority rule:

```
lower_CAN_identifier = higher_priority
```

Approximate frame transmission time:

```
frame_time = number_of_bits / bus_bit_rate
```

Worst-case waiting intuition:

```
waiting_time_for_low_priority_frame increases with higher_priority_traffic_load
```

A precise worst-case response-time analysis must include bit stuffing, frame size, blocking, retransmissions, and higher-priority interference. The lab simulation focuses on the arbitration principle.

### 4. Classroom Case Studies

Case Study A - Brake Message vs. Window Control:
A brake-pressure message should use a higher priority identifier than a power-window status message. During simultaneous transmission, the brake-related frame must win arbitration.

Case Study B - Diagnostic Traffic During Driving:
Diagnostic frames should not starve safety-critical frames. Poor identifier planning can make non-critical traffic interfere with time-critical control messages.

Case Study C - Bus Load Growth:
As new ECUs are added, bus load increases. Even if arbitration works correctly, low-priority messages may experience higher delay. This is why CAN network design includes bus-load budgeting.

### 5. Analysis Checklist

List all message identifiers, classify criticality, show the winning message, and explain why lower identifier wins. Include frame time and bus utilization if message length and bit rate are known.

### 6. Source Notes

- CAN arbitration and identifier-priority behavior are described in Bosch CAN documentation and vendor technical references: https://www.bosch-semiconductors.com/ip-modules/can-protocol/
- Python simulation uses deterministic comparison logic, not a physical CAN controller.


## Architecture

```text
CAN Nodes
  |-- Brake frame
  |-- Steering frame
  |-- Radar frame
  |-- Telemetry frame
          |
          v
Shared CAN Bus
  |-- simultaneous access attempt
  |-- bit-wise arbitration
          |
          v
Winner Selection
  |-- lowest identifier wins
          |
          v
Transmission Timeline
```

## Formulas and Required Theory

Approximate frame transmission time:

\[
T_{frame} = \frac{\text{frame bits}}{\text{bitrate}}
\]

For the simplified lab model:

\[
\text{frame bits} \approx 111 + 8 \times \text{payload bytes}
\]

Priority rule:

\[
\text{lower CAN ID} \Rightarrow \text{higher bus priority}
\]

## In-Lab Method

1. Define CAN frames with node name, CAN ID, release time, and payload size.
2. Add released frames to a pending queue.
3. Sort pending frames by CAN ID.
4. Transmit the lowest-ID frame first.
5. Record start and finish times.

In [1]:
frames = [
    {"node": "Brake", "can_id": 0x080, "release": 0, "bytes": 8},
    {"node": "Steering", "can_id": 0x120, "release": 0, "bytes": 8},
    {"node": "Telemetry", "can_id": 0x300, "release": 0, "bytes": 8},
    {"node": "Radar", "can_id": 0x100, "release": 1, "bytes": 8},
]
bitrate = 500_000
clock = 0.0
pending = []
sent = []
while frames or pending:
    pending.extend([f for f in frames if f["release"] <= clock])
    frames = [f for f in frames if f["release"] > clock]
    if not pending:
        clock = min(f["release"] for f in frames)
        continue
    pending.sort(key=lambda f: f["can_id"])
    frame = pending.pop(0)
    tx_ms = (111 + frame["bytes"] * 8) / bitrate * 1000
    start, finish = clock, clock + tx_ms
    sent.append((frame["node"], hex(frame["can_id"]), start, finish))
    clock = finish

print("EXP 6 - IN-LAB CAN ARBITRATION")
print(f"{'Node':12} {'CAN ID':>8} {'Start ms':>10} {'Finish ms':>10}")
for row in sent:
    print(f"{row[0]:12} {row[1]:>8} {row[2]:10.3f} {row[3]:10.3f}")

EXP 6 - IN-LAB CAN ARBITRATION
Node           CAN ID   Start ms  Finish ms
Brake            0x80      0.000      0.350
Steering        0x120      0.350      0.700
Telemetry       0x300      0.700      1.050
Radar           0x100      1.050      1.400


## Post-Lab Method

The post-lab cell ranks identifiers by priority and explains why safety messages should use lower identifiers than non-critical telemetry.

In [2]:
ids = [0x080, 0x100, 0x120, 0x300]
print("EXP 6 - POST-LAB PRIORITY EFFECT")
print("Lower CAN identifier wins arbitration.")
for can_id in sorted(ids):
    print(f"ID {hex(can_id):>5} priority rank {sorted(ids).index(can_id) + 1}")
print("Inference: safety messages should receive lower identifiers than telemetry.")

EXP 6 - POST-LAB PRIORITY EFFECT
Lower CAN identifier wins arbitration.
ID  0x80 priority rank 1
ID 0x100 priority rank 2
ID 0x120 priority rank 3
ID 0x300 priority rank 4
Inference: safety messages should receive lower identifiers than telemetry.


## What to Write in the Lab Record

- List each node and CAN ID.
- Show the arbitration order.
- Explain why brake and steering frames should have higher bus priority.
- Mention that this notebook uses a simplified frame-length model.

## References

- CAN arbitration explanation by HMS Networks: https://www.hms-networks.com/tech-blog/blogpost/hms-blog/2024/06/18/what-is-can-arbitration-and-how-does-this-work
- CAN bus arbitration overview: https://en.wikipedia.org/wiki/CAN_bus